# CardioVision: preprocessing and feature harmonization

Build train-only imputation/scaling pipelines, add Mean Arterial Pressure (MAP) where systolic and diastolic pressure are both available, align Cleveland to a binary target, and persist stratified train/test sets for in-dataset and zero-shot evaluation.

**Cross-dataset caution:** the requested shared names do not guarantee equivalent measurements. Cardio cholesterol/glucose are ordinal codes, Framingham cholesterol/glucose are measured values, and Cleveland has cholesterol in mg/dL and a binary fasting-glucose flag. Cleveland has systolic pressure only, so `MAP` uses resting systolic pressure as an explicitly imperfect proxy. Zero-shot scores therefore describe a deliberately difficult exploratory transfer, not clinically harmonized validation.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from IPython.display import display

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((p for p in candidates if (p / "data" / "cardio_train.csv").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate data/cardio_train.csv from the notebook working directory.")
DATA_DIR = PROJECT_ROOT / "data"
RESULT_DIR = PROJECT_ROOT / "result"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42
TEST_SIZE = 0.20
HARMONIZED_FEATURES = ["age_years", "sex", "chol", "glucose/fbs", "MAP", "smoke"]
cleveland_columns = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", "thalach",
    "exang", "oldpeak", "slope", "ca", "thal", "target",
]

cardio = pd.read_csv(DATA_DIR / "cardio_train.csv", sep=";")
framingham = pd.read_csv(DATA_DIR / "framingham.csv")
cleveland = pd.read_csv(
    DATA_DIR / "UCI" / "processed.cleveland.data",
    header=None,
    names=cleveland_columns,
    na_values=["?"],
)
for column in cleveland.columns:
    cleveland[column] = pd.to_numeric(cleveland[column], errors="coerce")

print("Loaded source shapes:", {
    "cardio_train": cardio.shape,
    "framingham": framingham.shape,
    "cleveland": cleveland.shape,
})

Loaded source shapes: {'cardio_train': (70000, 13), 'framingham': (4240, 16), 'cleveland': (303, 14)}


## Clinical features and binary targets

Sex is encoded consistently as 1 = male and 0 = female in the harmonized feature set. Cardio's categorical cholesterol and glucose values are retained as ordinal codes; Framingham uses measured total cholesterol and glucose; Cleveland uses total cholesterol and fasting blood sugar.

In [2]:
# Apply the same physiological BP bounds used during EDA to Cardio Train.
cardio_valid_bp = cardio["ap_hi"].between(50, 240, inclusive="both") & cardio["ap_lo"].between(30, 150, inclusive="both")
cardio = cardio.loc[cardio_valid_bp].copy()
cardio["age_years"] = (cardio["age"] / 365.25).astype(int)
cardio["sex"] = (cardio["gender"] == 2).astype(int)
cardio["MAP"] = cardio["ap_lo"] + (cardio["ap_hi"] - cardio["ap_lo"]) / 3
cardio["chol"] = cardio["cholesterol"]
cardio["glucose/fbs"] = cardio["gluc"]

framingham["age_years"] = framingham["age"]
framingham["sex"] = framingham["male"]
framingham["MAP"] = framingham["diaBP"] + (framingham["sysBP"] - framingham["diaBP"]) / 3
framingham["chol"] = framingham["totChol"]
framingham["glucose/fbs"] = framingham["glucose"]
framingham["smoke"] = framingham["currentSmoker"]

cleveland["age_years"] = cleveland["age"]
cleveland["chol"] = cleveland["chol"]
cleveland["glucose/fbs"] = cleveland["fbs"]
cleveland["smoke"] = np.nan  # This Cleveland file has no smoking field; impute from Cardio Train for zero-shot transfer.
cleveland["MAP"] = cleveland["trestbps"]  # Systolic-only proxy; true MAP cannot be derived without diastolic pressure.
cleveland["target"] = (cleveland["target"] > 0).astype(int)

sources = {
    "cardio_train": (cardio, "cardio"),
    "framingham": (framingham, "TenYearCHD"),
    "cleveland": (cleveland, "target"),
}

# Exclude identifiers, raw age, and duplicate source encodings from native features.
native_features = {
    "cardio_train": ["age_years", "sex", "height", "weight", "ap_hi", "ap_lo", "cholesterol", "gluc", "smoke", "alco", "active", "MAP"],
    "framingham": [column for column in framingham.columns if column not in {"TenYearCHD", "age", "male", "currentSmoker", "sysBP", "diaBP", "totChol", "glucose"}],
"cleveland": [column for column in cleveland.columns if column not in {"target", "age", "chol", "fbs", "smoke", "MAP"}],
}
native_features["framingham"] = list(dict.fromkeys(native_features["framingham"] + ["age_years", "sex", "currentSmoker", "sysBP", "diaBP", "totChol", "glucose", "MAP"]))
native_features["cleveland"] = list(dict.fromkeys(native_features["cleveland"] + ["age_years", "chol", "fbs"]))

print("Cardio Train records retained after BP filtering:", len(cardio))
print("Cleveland original target values mapped to binary:", sorted(cleveland["target"].unique()))
print("Native feature counts:", {name: len(columns) for name, columns in native_features.items()})
print("Harmonized feature order:", HARMONIZED_FEATURES)

Cardio Train records retained after BP filtering: 68761
Cleveland original target values mapped to binary: [np.int64(0), np.int64(1)]
Native feature counts: {'cardio_train': 12, 'framingham': 19, 'cleveland': 14}
Harmonized feature order: ['age_years', 'sex', 'chol', 'glucose/fbs', 'MAP', 'smoke']


## Stratified splitting, train-only imputation/scaling, and persistence

Each dataset receives its own stratified 80/20 split and native scaler. For zero-shot evaluation, destination missing values and scaling use only the Cardio Train harmonized training-set imputer/scaler; no destination data is used to fit the transferred model's preprocessing.

In [3]:
arrays = {}
split_frames = {}
harmonized_raw = {}
preprocessors = {}
split_report = []

for name, (df, target_column) in sources.items():
    y = df[target_column].astype(int).to_numpy()
    native = df[native_features[name]].apply(pd.to_numeric, errors="coerce")
    common = df[HARMONIZED_FEATURES].apply(pd.to_numeric, errors="coerce")
    row_indices = np.arange(len(df))
    train_idx, test_idx = train_test_split(
        row_indices,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y,
    )

    # Fit both imputers and scalers on the training fold only.
    native_imputer = SimpleImputer(strategy="median")
    X_train_native = native_imputer.fit_transform(native.iloc[train_idx])
    X_test_native = native_imputer.transform(native.iloc[test_idx])
    native_scaler = StandardScaler()
    X_train_native = native_scaler.fit_transform(X_train_native)
    X_test_native = native_scaler.transform(X_test_native)

    common_imputer = SimpleImputer(strategy="median", keep_empty_features=True)
    X_train_common_unscaled = common_imputer.fit_transform(common.iloc[train_idx])
    X_test_common_unscaled = common_imputer.transform(common.iloc[test_idx])
    common_scaler = StandardScaler()
    X_train_common = common_scaler.fit_transform(X_train_common_unscaled)
    X_test_common = common_scaler.transform(X_test_common_unscaled)

    arrays[f"{name}_X_train"] = X_train_native
    arrays[f"{name}_X_test"] = X_test_native
    arrays[f"{name}_y_train"] = y[train_idx]
    arrays[f"{name}_y_test"] = y[test_idx]
    arrays[f"{name}_feature_names"] = np.asarray(native_features[name], dtype=str)
    arrays[f"{name}_harm_X_train"] = X_train_common
    arrays[f"{name}_harm_X_test"] = X_test_common
    arrays[f"{name}_harm_feature_names"] = np.asarray(HARMONIZED_FEATURES, dtype=str)
    arrays[f"{name}_row_indices_train"] = train_idx
    arrays[f"{name}_row_indices_test"] = test_idx
    split_frames[name] = (train_idx, test_idx)
    harmonized_raw[name] = common
    preprocessors[name] = (native_imputer, native_scaler, common_imputer, common_scaler)

    split_report.append({
        "dataset": name,
        "rows": len(df),
        "train": len(train_idx),
        "test": len(test_idx),
        "native_features": len(native_features[name]),
        "target_train_positive_rate": float(y[train_idx].mean()),
        "target_test_positive_rate": float(y[test_idx].mean()),
    })

# Strict zero-shot preprocessing: transform destination test features with cardio-train fits only.
cardio_imputer = preprocessors["cardio_train"][2]
cardio_scaler = preprocessors["cardio_train"][3]
arrays["cardio_train_harm_X_train_ood_model"] = arrays["cardio_train_harm_X_train"]
arrays["cardio_train_harm_X_test_ood_model"] = arrays["cardio_train_harm_X_test"]
for name in ["framingham", "cleveland"]:
    _, test_idx = split_frames[name]
    destination_test = harmonized_raw[name].iloc[test_idx]
    destination_test_imputed = cardio_imputer.transform(destination_test)
    arrays[f"{name}_ood_X_test"] = cardio_scaler.transform(destination_test_imputed)
    arrays[f"{name}_ood_y_test"] = sources[name][0][sources[name][1]].astype(int).to_numpy()[test_idx]

output_path = RESULT_DIR / "preprocessed_data.npz"
np.savez_compressed(output_path, **arrays)
split_report_df = pd.DataFrame(split_report).set_index("dataset")
display(split_report_df)
print(f"Saved train-only-imputed, train-scaled arrays to {output_path}")
print("All train/test array shapes:", {
    key: value.shape for key, value in arrays.items()
    if key.endswith(("X_train", "X_test", "ood_X_test"))
})

,rows,train,test,native_features,target_train_positive_rate,target_test_positive_rate
dataset,,,,,,
cardio_train,68761,55008,13753,12,0.494801,0.494801
framingham,4240,3392,848,19,0.151828,0.152123
cleveland,303,242,61,14,0.458678,0.459016


Saved train-only-imputed, train-scaled arrays to C:\Users\kunug\OneDrive\Desktop\cardioVision\result\preprocessed_data.npz
All train/test array shapes: {'cardio_train_X_train': (55008, 12), 'cardio_train_X_test': (13753, 12), 'cardio_train_harm_X_train': (55008, 6), 'cardio_train_harm_X_test': (13753, 6), 'framingham_X_train': (3392, 19), 'framingham_X_test': (848, 19), 'framingham_harm_X_train': (3392, 6), 'framingham_harm_X_test': (848, 6), 'cleveland_X_train': (242, 14), 'cleveland_X_test': (61, 14), 'cleveland_harm_X_train': (242, 6), 'cleveland_harm_X_test': (61, 6), 'framingham_ood_X_test': (848, 6), 'cleveland_ood_X_test': (61, 6)}
